# Z+J development compile — 2026-10-11
Target: first-window `localMass = 0` and `c1 > 0` imply `localAngularMoment > 0`.
Only pinned source → fresh Z01 → Z02 → Z+J. No C04/Z04 recompile, installation, `lake update`, commit, push, claim admission, or epsilon/existence/uniqueness/Q claim.
This notebook transports frozen input bytes; it is not proof authority. Missing environment stops before execution. Preserve the resulting archive under the repository's `colab/evidence/` on D: and recheck its SHA-256 before receipt review.


In [ ]:
import base64, hashlib, json, os, subprocess, sys
from datetime import datetime, timezone
from pathlib import Path

# Frozen execution transport, not an authority receipt; no installer/lake update.
PAYLOAD = json.loads(r'''[{"sha256":"ff7101067a98fbaa8974c1aa409b83740cd17eb4f1778f20cdc23b44e2a5c267","base64":"aW1wb3J0IE5hdmllclN0b2tlcy5PdXRnb2luZ1Byb2ZpbGUKCi8tIQojIFAyIFotMDFEOiBwYXJ0aWFsIG1hc3MgYW5kIHRhaWwgaWRlbnRpdGllcwoKVGhlIHR3byBuYW1lZCBwcm9wb3NpdGlvbnMgY29uY2VybiB0aGUgZXhhY3Qgc291cmNlIGBtYXNzTW9tZW50YCwgZXZhbHVhdGVkIGF0CmBwdWxzZVN0YXJ0ICsgbG9nIHJgLCBmb3IgdGhlIGFtcGxpdHVkZSBvZiBvbmUgZml4ZWQgc291cmNlIFByb2ZpbGUuIFRoZSBwb3NpdGl2ZQpgbW9tZW50U2NhbGUgKiBzaGFwZWAgaXMgYW4gaW50ZWdyYWwgcHJlZmFjdG9yLCBub3QgYSBjb252ZXJzaW9uIGZhY3RvciBiZXR3ZWVuCmBtYXNzTW9tZW50YCBhbmQgYFByb2ZpbGUuTWAuCgpObyBib3VuZGFyeSBzaWduLCBidW1wLWludGVncmFsIHBvc2l0aXZpdHksIElWVCwgb3IgbWFzcy16ZXJvIGV4aXN0ZW5jZSBpcwpwcm92ZWQgaGVyZS4gRGV2ZWxvcG1lbnQgY29tcGlsYXRpb24gaXMgc2VwYXJhdGUgZnJvbSByZWNlaXB0IGFkbWlzc2lvbi4KLS8KCm9wZW4gU2V0IE1lYXN1cmVUaGVvcnkKb3BlbiBOYXZpZXJTdG9rZXMuT3V0Z29pbmdTY2hlZHVsZSBOYXZpZXJTdG9rZXMuT3V0Z29pbmdQcm9maWxlCgpuYW1lc3BhY2UgTlNSVy5QMgoKdGhlb3JlbSBsb2NhbE1hc3NfcGFydGlhbF9pZGVudGl0eSAoRiA6IFByb2ZpbGUpIChldGEgciA6IOKEnSkKICAgIChociA6IDEg4omkIHIpIChfaHJfZW5kIDogciDiiaQgUmVhbC5leHAgRi5kYXRhLmNvcmUucHVsc2VMZW5ndGgpIDoKICAgIG1hc3NNb21lbnQgRi5kYXRhLmNvcmUgRi5hbXAgZXRhIChGLmRhdGEuY29yZS5wdWxzZVN0YXJ0ICsgUmVhbC5sb2cgcikgPQogICAgICBwcmVmaXhNIEYuZGF0YS5jb3JlICogZXRhICsKICAgICAgICAobW9tZW50U2NhbGUgRi5kYXRhLmNvcmUgMCAqIHNoYXBlIGV0YSkgKgogICAgICAgICAgKOKIqyB4IGluICgxIDog4oSdKS4uciwKICAgICAgICAgICAgeCBeIEYuZGF0YS5jb3JlLmV4cG9uZW50cyAwICogcmFkaWFsUHVsc2UgRi5kYXRhLmNvcmUgRi5hbXAgZXRhIHgpIDo9IGJ5CiAgbGV0IGMgOj0gRi5kYXRhLmNvcmUKICBoYXZlIGhyX3BvcyA6IDAgPCByIDo9IGx0X29mX2x0X29mX2xlIHplcm9fbHRfb25lIGhyCiAgaGF2ZSBobG9nIDogMCDiiaQgUmVhbC5sb2cgciA6PSBSZWFsLmxvZ19ub25uZWcgaHIKICBoYXZlIGhGIDogQ29udGludW91cyAoZnVuIHQgPT4gUmVhbC5leHAgdCAqIGF4aWFsIGMgRi5hbXAgKHQsIGV0YSkpIDo9CiAgICBSZWFsLmNvbnRpbnVvdXNfZXhwLm11bCAoYXhpYWxfcmFkaWFsX2NvbnREaWZmIGMgRi5hbXAgZXRhKS5jb250aW51b3VzCiAgaGF2ZSBoc3BsaXQgOj0gaW50ZXJ2YWxJbnRlZ3JhbC5pbnRlZ3JhbF9hZGRfYWRqYWNlbnRfaW50ZXJ2YWxzICjOvCA6PSB2b2x1bWUpCiAgICAoaEYuaW50ZXJ2YWxJbnRlZ3JhYmxlIDAgYy5wdWxzZVN0YXJ0KQogICAgKGhGLmludGVydmFsSW50ZWdyYWJsZSBjLnB1bHNlU3RhcnQgKGMucHVsc2VTdGFydCArIFJlYWwubG9nIHIpKQogIGhhdmUgaEkgOiAo4oirIHQgaW4gYy5wdWxzZVN0YXJ0Li5jLnB1bHNlU3RhcnQgKyBSZWFsLmxvZyByLAogICAgICBSZWFsLmV4cCB0ICogYXhpYWwgYyBGLmFtcCAodCwgZXRhKSkgPQogICAgICAobW9tZW50U2NhbGUgYyAwICogc2hhcGUgZXRhKSAqCiAgICAgICAgKOKIqyB4IGluICgxIDog4oSdKS4uciwgeCBeIGMuZXhwb25lbnRzIDAgKiByYWRpYWxQdWxzZSBjIEYuYW1wIGV0YSB4KSA6PSBieQogICAgY2FsYwogICAgICBfID0g4oirIHQgaW4gYy5wdWxzZVN0YXJ0Li5jLnB1bHNlU3RhcnQgKyBSZWFsLmxvZyByLAogICAgICAgICAgKG1vbWVudFNjYWxlIGMgMCAqIHNoYXBlIGV0YSkgKgogICAgICAgICAgICAoUmVhbC5leHAgKChjLmV4cG9uZW50cyAwICsgMSkgKiAodCAtIGMucHVsc2VTdGFydCkpICoKICAgICAgICAgICAgICByYWRpYWxQdWxzZSBjIEYuYW1wIGV0YSAoUmVhbC5leHAgKHQgLSBjLnB1bHNlU3RhcnQpKSkgOj0gYnkKICAgICAgICBhcHBseSBpbnRlcnZhbEludGVncmFsLmludGVncmFsX2NvbmdyCiAgICAgICAgaW50cm8gdCBodAogICAgICAgIGFwcGx5IG1hc3NfaW50ZWdyYW5kX3B1bHNlIGMgRi5hbXAgZXRhCiAgICAgICAgZXhhY3QgKHVJY2Nfb2ZfbGUgKHNob3cgYy5wdWxzZVN0YXJ0IOKJpCBjLnB1bHNlU3RhcnQgKyBSZWFsLmxvZyByIGJ5CiAgICAgICAgICBsaW5hcml0aCkg4pa4IGh0KS4xCiAgICAgIF8gPSBfIDo9IGJ5CiAgICAgICAgcncgW2ludGVydmFsSW50ZWdyYWwuaW50ZWdyYWxfY29uc3RfbXVsXQogICAgICAgIHJ3IFtleHBfd2VpZ2h0X3N1YnN0aXR1dGlvbiAocmFkaWFsUHVsc2VfY29udGludW91c09uIGMgRi5hbXAgZXRhKQogICAgICAgICAgKGMuZXhwb25lbnRzIDApIGMucHVsc2VTdGFydCAoUmVhbC5sb2cgcildCiAgICAgICAgcncgW1JlYWwuZXhwX2xvZyBocl9wb3NdCiAgY2hhbmdlIG1hc3NNb21lbnQgYyBGLmFtcCBldGEgKGMucHVsc2VTdGFydCArIFJlYWwubG9nIHIpID0gXwogIGNhbGMKICAgIF8gPSBtYXNzTW9tZW50IGMgRi5hbXAgZXRhIGMucHVsc2VTdGFydCArCiAgICAgICAgKOKIqyB0IGluIGMucHVsc2VTdGFydC4uYy5wdWxzZVN0YXJ0ICsgUmVhbC5sb2cgciwKICAgICAgICAgIFJlYWwuZXhwIHQgKiBheGlhbCBjIEYuYW1wICh0LCBldGEpKSA6PSBieQogICAgICB1bmZvbGQgbWFzc01vbWVudAogICAgICBydyBb4oaQIGhzcGxpdF0KICAgICAgcmluZwogICAgXyA9IF8gOj0gYnkgcncgW21hc3NNb21lbnRfYXRfcHVsc2VTdGFydCwgaEldCgp0aGVvcmVtIGxvY2FsTWFzc190YWlsX2lkZW50aXR5IChGIDogUHJvZmlsZSkgKGV0YSByIDog4oSdKQogICAgKGhyIDogMSDiiaQgcikgKGhyX2VuZCA6IHIg4omkIFJlYWwuZXhwIEYuZGF0YS5jb3JlLnB1bHNlTGVuZ3RoKSA6CiAgICBtYXNzTW9tZW50IEYuZGF0YS5jb3JlIEYuYW1wIGV0YSAoRi5kYXRhLmNvcmUucHVsc2VTdGFydCArIFJlYWwubG9nIHIpID0KICAgICAgLShtb21lbnRTY2FsZSBGLmRhdGEuY29yZSAwICogc2hhcGUgZXRhKSAqCiAgICAgICAgKOKIqyB4IGluIHIuLlJlYWwuZXhwIEYuZGF0YS5jb3JlLnB1bHNlTGVuZ3RoLAogICAgICAgICAgeCBeIEYuZGF0YS5jb3JlLmV4cG9uZW50cyAwICogcmFkaWFsUHVsc2UgRi5kYXRhLmNvcmUgRi5hbXAgZXRhIHgpIDo9IGJ5CiAgbGV0IGMgOj0gRi5kYXRhLmNvcmUKICBsZXQgZyA6IOKEnSDihpIg4oSdIDo9IGZ1biB4ID0+IHggXiBjLmV4cG9uZW50cyAwICogcmFkaWFsUHVsc2UgYyBGLmFtcCBldGEgeAogIGhhdmUgaGcgOiBDb250aW51b3VzT24gZyAoSWNjIDEgKFJlYWwuZXhwIGMucHVsc2VMZW5ndGgpKSA6PSBieQogICAgYXBwbHkgKGNvbnRpbnVvdXNPbl9pZC5ycG93X2NvbnN0IChmdW4geCBoeCA9PgogICAgICBPci5pbmwgKG5lX29mX2d0IChsdF9vZl9sdF9vZl9sZSB6ZXJvX2x0X29uZSBoeC4xKSkpKS5tdWwKICAgIGV4YWN0IChyYWRpYWxQdWxzZV9jb250aW51b3VzT24gYyBGLmFtcCBldGEpLm1vbm8KICAgICAgKGZ1biB4IGh4ID0+IGx0X29mX2x0X29mX2xlIHplcm9fbHRfb25lIGh4LjEpCiAgaGF2ZSBoZ19sZWZ0IDogSW50ZXJ2YWxJbnRlZ3JhYmxlIGcgdm9sdW1lIDEgciA6PSBieQogICAgYXBwbHkgQ29udGludW91c09uLmludGVydmFsSW50ZWdyYWJsZQogICAgcncgW3VJY2Nfb2ZfbGUgaHJdCiAgICBleGFjdCBoZy5tb25vIChmdW4geCBoeCA9PiDin6hoeC4xLCBoeC4yLnRyYW5zIGhyX2VuZOKfqSkKICBoYXZlIGhnX3JpZ2h0IDogSW50ZXJ2YWxJbnRlZ3JhYmxlIGcgdm9sdW1lIHIgKFJlYWwuZXhwIGMucHVsc2VMZW5ndGgpIDo9IGJ5CiAgICBhcHBseSBDb250aW51b3VzT24uaW50ZXJ2YWxJbnRlZ3JhYmxlCiAgICBydyBbdUljY19vZl9sZSBocl9lbmRdCiAgICBleGFjdCBoZy5tb25vIChmdW4geCBoeCA9PiDin6hoci50cmFucyBoeC4xLCBoeC4y4p+pKQogIGhhdmUgaHNwbGl0IDo9IGludGVydmFsSW50ZWdyYWwuaW50ZWdyYWxfYWRkX2FkamFjZW50X2ludGVydmFscyBoZ19sZWZ0IGhnX3JpZ2h0CiAgaGF2ZSBoY2FuY2VsIDo9IChwdWxzZV9jbG9zZXNfcHJlZml4IGMgRi5hbXAgZXRhKS4xCiAgcncgW2xvY2FsTWFzc19wYXJ0aWFsX2lkZW50aXR5IEYgZXRhIHIgaHIgaHJfZW5kXQogIGNoYW5nZSBwcmVmaXhNIGMgKiBldGEgKyAobW9tZW50U2NhbGUgYyAwICogc2hhcGUgZXRhKSAqICjiiKsgeCBpbiAxLi5yLCBnIHgpID0KICAgIC0obW9tZW50U2NhbGUgYyAwICogc2hhcGUgZXRhKSAqICjiiKsgeCBpbiByLi5SZWFsLmV4cCBjLnB1bHNlTGVuZ3RoLCBnIHgpCiAgY2hhbmdlIHByZWZpeE0gYyAqIGV0YSArIChtb21lbnRTY2FsZSBjIDAgKiBzaGFwZSBldGEpICoKICAgICjiiKsgeCBpbiAoMSA6IOKEnSkuLlJlYWwuZXhwIGMucHVsc2VMZW5ndGgsIGcgeCkgPSAwIGF0IGhjYW5jZWwKICBydyBb4oaQIGhzcGxpdF0gYXQgaGNhbmNlbAogIG5saW5hcml0aCBbaGNhbmNlbF0KCiNwcmludCBheGlvbXMgTlNSVy5QMi5sb2NhbE1hc3NfcGFydGlhbF9pZGVudGl0eQojcHJpbnQgYXhpb21zIE5TUlcuUDIubG9jYWxNYXNzX3RhaWxfaWRlbnRpdHkKCmVuZCBOU1JXLlAyCg==","name":"P2_Z01_LocalMassIdentities.lean"},{"sha256":"7b5764911790054cbba982acfde8ed673705a2d68e85bcd341507d7e0b5f8c2c","base64":"aW1wb3J0IFAyX1owMV9Mb2NhbE1hc3NJZGVudGl0aWVzCgovLSEKIyBQMiBaLTAyOiBjb2VmZmljaWVudC10by1tYXNzIGJvdW5kYXJ5IHNpZ25zCgpEZXZlbG9wbWVudCBpbXBsZW1lbnRhdGlvbiwgbm90IGNvbXBpbGVkIG9yIGFkbWl0dGVkIHlldC4gVGhlIG9ubHkgZXh0ZXJuYWwKcHJvb2YgZGVwZW5kZW5jeSBpcyBaMDEuIEMwNCBhbmQgdGhlIEYxLS1DMDQgZXh0ZXJuYWwgY2hhaW4gYXJlIG5vdCBpbXBvcnRlZC4KCmBsb2NhbE1hc3NgIGlzIHRoZSBleGFjdCBzb3VyY2UgbWFzc01vbWVudCBhdCBCICsgbG9nIHIuIFRoZSBjb2VmZmljaWVudCBhbGlhcwpiZWxvdyBpcyB0aGUgYWN0dWFsIHNlY29uZCBBWElBTCByZXBhaXIgY29lZmZpY2llbnQgb2YgdGhlIHNhbWUgUHJvZmlsZSwgbm90CnRoZSBhbmd1bGFyLXJlc2V0IGNvZWZmaWNpZW50cyBzdG9yZWQgaW4gRi5yZXNldC4gSXRzIHBvc2l0aXZpdHkgaXMgYW4gZXhwbGljaXQKaHlwb3RoZXNpcyBvZiB0aGUgdXBwZXItc2lnbiB0aGVvcmVtLiBObyBzbWFsbC1ldGEgZXhpc3RlbmNlLCBJVlQsIG1hc3MtemVybwpleGlzdGVuY2UsIHVuaXF1ZW5lc3MsIG9yIGZpcnN0LXplcm8gcHJvcG9zaXRpb24gaXMgYXNzZXJ0ZWQgaGVyZS4KLS8KCm9wZW4gU2V0IE1lYXN1cmVUaGVvcnkKb3BlbiBOYXZpZXJTdG9rZXMuT3V0Z29pbmdTY2hlZHVsZSBOYXZpZXJTdG9rZXMuT3V0Z29pbmdQcm9maWxlCm9wZW4gTmF2aWVyU3Rva2VzLk91dGdvaW5nUHVsc2VCb3VuZHMKCm5hbWVzcGFjZSBOU1JXLlAyCgpub25jb21wdXRhYmxlIGRlZiBsb2NhbE1hc3MgKEYgOiBQcm9maWxlKSAoZXRhIHIgOiDihJ0pIDog4oSdIDo9CiAgbWFzc01vbWVudCBGLmRhdGEuY29yZSBGLmFtcCBldGEgKEYuZGF0YS5jb3JlLnB1bHNlU3RhcnQgKyBSZWFsLmxvZyByKQoKbm9uY29tcHV0YWJsZSBkZWYgbG9jYWxTZWNvbmRSZXBhaXJDb2VmZmljaWVudCAoRiA6IFByb2ZpbGUpIChldGEgOiDihJ0pIDog4oSdIDo9CiAgTmF2aWVyU3Rva2VzLkxvY2FsaXplZE1vbWVudFJlcGFpci5jb2VmZmljaWVudHMKICAgIEYuZGF0YS5jb3JlLmV4cG9uZW50cyBGLmRhdGEuY29yZS5sb3dlciBGLmRhdGEuY29yZS51cHBlcgogICAgKGRlYnQgRi5kYXRhLmNvcmUgRi5hbXAgZXRhKSAoMSA6IEZpbiAyKQoKdGhlb3JlbSBwcmVmaXhNX3BvcyAoYyA6IFBhcmFtZXRlcnMpIDogMCA8IHByZWZpeE0gYyA6PSBieQogIGhhdmUgaGkgOiAwIOKJpCDiiKsgeSBpbiAoMCA6IOKEnSkuLmMucHVsc2VTdGFydCwKICAgICAgUmVhbC5leHAgeSAqIGRyb3BDb2VmZmljaWVudCBjLm0geSA6PQogICAgaW50ZXJ2YWxJbnRlZ3JhbC5pbnRlZ3JhbF9ub25uZWdfb2ZfZm9yYWxsIGMucHVsc2VTdGFydF9wb3MubGUKICAgICAgKGZ1biB5ID0+IG11bF9ub25uZWcgKFJlYWwuZXhwX3BvcyB5KS5sZSAoZHJvcENvZWZmaWNpZW50X2JvdW5kcyBjLm0geSkuMSkKICB1bmZvbGQgcHJlZml4TQogIGxpbmFyaXRoCgpwcml2YXRlIHRoZW9yZW0gZmlyc3RfbG93ZXJfbGVfbG93ZXIgKGMgOiBQYXJhbWV0ZXJzKSAoaSA6IEZpbiAyKSA6CiAgICBjLmxvd2VyIDAg4omkIGMubG93ZXIgaSA6PSBieQogIGZpbl9jYXNlcyBpCiAgwrcgZXhhY3QgbGVfcmZsCiAgwrcgZXhhY3QgKGMubG93ZXJfbHRfdXBwZXIgMCkubGUudHJhbnMgKGMuaW50ZXJ2YWxzX3NlcGFyYXRlZCAwIDEgKGJ5IGRlY2lkZSkpCgpwcml2YXRlIHRoZW9yZW0gZmlyc3RfdXBwZXJfZ3Rfb25lIChjIDogUGFyYW1ldGVycykgOiAxIDwgYy51cHBlciAwIDo9CiAgKGMub25lX2x0X2xvd2VyIDApLnRyYW5zIChjLmxvd2VyX2x0X3VwcGVyIDApCgpwcml2YXRlIHRoZW9yZW0gZmlyc3RfYnVtcF96ZXJvX2FmdGVyIChjIDogUGFyYW1ldGVycykgKHggOiDihJ0pCiAgICAoaHggOiBjLnVwcGVyIDAg4omkIHgpIDoKICAgIE5hdmllclN0b2tlcy5Mb2NhbGl6ZWRNb21lbnRSZXBhaXIuYnVtcCAoYy5sb3dlciAwKSAoYy51cHBlciAwKSB4ID0gMCA6PSBieQogIGJ5X2NvbnRyYSBoYgogIGhhdmUgaHMgOj0gKE5hdmllclN0b2tlcy5Mb2NhbGl6ZWRNb21lbnRSZXBhaXIuYnVtcF90c3VwcG9ydF9zdWJzZXRfb3BlbgogICAgKGMubG93ZXIgMCkgKGMudXBwZXIgMCkgKGMubG93ZXJfbHRfdXBwZXIgMCkpIChzdWJzZXRfdHN1cHBvcnQgXyBoYikKICBleGFjdCAobm90X2x0X29mX2dlIGh4KSBocy4yCgp0aGVvcmVtIHJhZGlhbFB1bHNlX2JlZm9yZV9maXJzdF9sb3dlciAoRiA6IFByb2ZpbGUpIChldGEgeCA6IOKEnSkKICAgIChoeCA6IHgg4omkIEYuZGF0YS5jb3JlLmxvd2VyIDApIDoKICAgIHJhZGlhbFB1bHNlIEYuZGF0YS5jb3JlIEYuYW1wIGV0YSB4ID0KICAgICAgRi5hbXAgZXRhICogbWFpblB1bHNlIChGLmRhdGEuY29yZS5sYW0gKiBSZWFsLmxvZyB4KSA6PSBieQogIGxldCBjIDo9IEYuZGF0YS5jb3JlCiAgaGF2ZSBoYyA6IGNvcnJlY3Rpb24gYyBGLmFtcCBldGEgeCA9IDAgOj0gYnkKICAgIGFwcGx5IGNvcnJlY3Rpb25femVyb19vZl9vdXRzaWRlCiAgICBpbnRybyBpIGhpCiAgICBoYXZlIGhsbyA6PSBmaXJzdF9sb3dlcl9sZV9sb3dlciBjIGkKICAgIGV4YWN0IChub3RfbHRfb2ZfZ2UgKGh4LnRyYW5zIGhsbykpIGhpLjEKICBjaGFuZ2UgRi5hbXAgZXRhICogbWFpblB1bHNlIChjLmxhbSAqIFJlYWwubG9nIHgpICsgY29ycmVjdGlvbiBjIEYuYW1wIGV0YSB4ID0gXwogIHJ3IFtoYywgYWRkX3plcm9dCgp0aGVvcmVtIGxvY2FsTWFzc19sb3dlcl9wb3MgKEYgOiBQcm9maWxlKSAoZXRhIDog4oSdKSAoaGV0YSA6IDAgPCBldGEpIDoKICAgIDAgPCBsb2NhbE1hc3MgRiBldGEgKEYuZGF0YS5jb3JlLmxvd2VyIDApIDo9IGJ5CiAgbGV0IGMgOj0gRi5kYXRhLmNvcmUKICBoYXZlIGhsIDogMSDiiaQgYy5sb3dlciAwIDo9IChjLm9uZV9sdF9sb3dlciAwKS5sZQogIGhhdmUgaGxfZW5kIDogYy5sb3dlciAwIOKJpCBSZWFsLmV4cCBjLnB1bHNlTGVuZ3RoIDo9CiAgICAoYy5sb3dlcl9sdF91cHBlciAwKS5sZS50cmFucyAoYy51cHBlcl9sdF9lbmQgMCkubGUKICBoYXZlIGhhbXAgOiAwIDwgRi5hbXAgZXRhIDo9CiAgICBOYXZpZXJTdG9rZXMuQ29ycmVjdGVkUHVsc2VBbXBsaXR1ZGUuYW1wbGl0dWRlX3BvcyBGLmRhdGEgRi5yZXNldC5jb2VmZmljaWVudHMgZXRhCiAgaGF2ZSBoaSA6IDAg4omkIOKIqyB4IGluICgxIDog4oSdKS4uYy5sb3dlciAwLAogICAgICB4IF4gYy5leHBvbmVudHMgMCAqIHJhZGlhbFB1bHNlIGMgRi5hbXAgZXRhIHggOj0gYnkKICAgIGFwcGx5IGludGVydmFsSW50ZWdyYWwuaW50ZWdyYWxfbm9ubmVnIGhsCiAgICBpbnRybyB4IGh4CiAgICBydyBbcmFkaWFsUHVsc2VfYmVmb3JlX2ZpcnN0X2xvd2VyIEYgZXRhIHggaHguMl0KICAgIGV4YWN0IG11bF9ub25uZWcgKFJlYWwucnBvd19ub25uZWcgKGJ5IGxpbmFyaXRoIFtoeC4xXSkgXykKICAgICAgKG11bF9ub25uZWcgaGFtcC5sZSAoTmF2aWVyU3Rva2VzLlB1bHNlQW1wbGl0dWRlLm1haW5QdWxzZV9ub25uZWcKICAgICAgICAobXVsX25vbm5lZyBjLmxhbV9wb3MubGUgKFJlYWwubG9nX25vbm5lZyBoeC4xKSkpKQogIGhhdmUgaHAgOj0gbXVsX3BvcyAocHJlZml4TV9wb3MgYykgaGV0YQogIGhhdmUgaGsgOj0gbXVsX3BvcyAobW9tZW50U2NhbGVfcG9zIGMgMCkgKHNoYXBlX3BvcyBldGEpCiAgY2hhbmdlIDAgPCBtYXNzTW9tZW50IGMgRi5hbXAgZXRhIChjLnB1bHNlU3RhcnQgKyBSZWFsLmxvZyAoYy5sb3dlciAwKSkKICBydyBbbG9jYWxNYXNzX3BhcnRpYWxfaWRlbnRpdHkgRiBldGEgKGMubG93ZXIgMCkgaGwgaGxfZW5kXQogIGV4YWN0IGFkZF9wb3Nfb2ZfcG9zX29mX25vbm5lZyBocCAobXVsX25vbm5lZyBoay5sZSBoaSkKCnRoZW9yZW0gc2Vjb25kQnVtcFdlaWdodGVkTW9tZW50X3BvcyAoYyA6IFBhcmFtZXRlcnMpIDoKICAgIDAgPCDiiKsgeCA6IOKEnSwgeCBeIGMuZXhwb25lbnRzIDAgKgogICAgICBOYXZpZXJTdG9rZXMuTG9jYWxpemVkTW9tZW50UmVwYWlyLmJ1bXAgKGMubG93ZXIgMSkgKGMudXBwZXIgMSkgeCA6PSBieQogIGhhdmUgaGVudHJ5IDo9IGFjdHVhbF9tYXRyaXhfZW50cnkgYyAoMCA6IEZpbiAyKSAoMSA6IEZpbiAyKQogIGNoYW5nZSAo4oirIHggOiDihJ0sIHggXiBjLmV4cG9uZW50cyAwICoKICAgIE5hdmllclN0b2tlcy5Mb2NhbGl6ZWRNb21lbnRSZXBhaXIuYnVtcCAoYy5sb3dlciAxKSAoYy51cHBlciAxKSB4KSA9IF8gYXQgaGVudHJ5CiAgcncgW2hlbnRyeV0KICBleGFjdCBtdWxfcG9zIChyb3dNb21lbnRfcG9zIGMgMCkgKFJlYWwuZXhwX3BvcyBfKQoKdGhlb3JlbSB1cHBlclRhaWxfcmFkaWFsUHVsc2VfZXEgKEYgOiBQcm9maWxlKSAoZXRhIHggOiDihJ0pCiAgICAoaHggOiBGLmRhdGEuY29yZS51cHBlciAwIOKJpCB4KQogICAgKF9oeF9lbmQgOiB4IOKJpCBSZWFsLmV4cCBGLmRhdGEuY29yZS5wdWxzZUxlbmd0aCkgOgogICAgcmFkaWFsUHVsc2UgRi5kYXRhLmNvcmUgRi5hbXAgZXRhIHggPQogICAgICBsb2NhbFNlY29uZFJlcGFpckNvZWZmaWNpZW50IEYgZXRhICoKICAgICAgICBOYXZpZXJTdG9rZXMuTG9jYWxpemVkTW9tZW50UmVwYWlyLmJ1bXAKICAgICAgICAgIChGLmRhdGEuY29yZS5sb3dlciAxKSAoRi5kYXRhLmNvcmUudXBwZXIgMSkgeCA6PSBieQogIGxldCBjIDo9IEYuZGF0YS5jb3JlCiAgaGF2ZSBoZXhwIDogUmVhbC5leHAgKDExIC8gYy5sYW0pIOKJpCB4IDo9CiAgICAoYy5tYWluX2VuZF9sdF9sb3dlciAwKS5sZS50cmFucyAoKGMubG93ZXJfbHRfdXBwZXIgMCkubGUudHJhbnMgaHgpCiAgaGF2ZSBobG9nIDogMTEgLyBjLmxhbSDiiaQgUmVhbC5sb2cgeCA6PSBieQogICAgaGF2ZSBoIDo9IFJlYWwubG9nX2xlX2xvZyAoUmVhbC5leHBfcG9zICgxMSAvIGMubGFtKSkgaGV4cAogICAgc2ltcGEgb25seSBbUmVhbC5sb2dfZXhwXSB1c2luZyBoCiAgaGF2ZSBoYXJnIDogMTEg4omkIGMubGFtICogUmVhbC5sb2cgeCA6PSBieQogICAgaGF2ZSBoIDo9IG11bF9sZV9tdWxfb2Zfbm9ubmVnX2xlZnQgaGxvZyBjLmxhbV9wb3MubGUKICAgIGhhdmUgaGlkIDogYy5sYW0gKiAoMTEgLyBjLmxhbSkgPSAxMSA6PSBieSBmaWVsZF9zaW1wIFtjLmxhbV9wb3MubmUnXQogICAgcndhIFtoaWRdIGF0IGgKICBoYXZlIGhtIDo9IG1haW5QdWxzZV96ZXJvX3JpZ2h0IGhhcmcKICBoYXZlIGhiIDo9IGZpcnN0X2J1bXBfemVyb19hZnRlciBjIHggaHgKICBjaGFuZ2UgRi5hbXAgZXRhICogbWFpblB1bHNlIChjLmxhbSAqIFJlYWwubG9nIHgpICsgY29ycmVjdGlvbiBjIEYuYW1wIGV0YSB4ID0gXwogIHJ3IFtobSwgbXVsX3plcm8sIHplcm9fYWRkXQogIHNpbXAgb25seSBbY29ycmVjdGlvbiwgTmF2aWVyU3Rva2VzLkxvY2FsaXplZE1vbWVudFJlcGFpci5yZXBhaXIsIEZpbi5zdW1fdW5pdl90d29dCiAgcncgW2hiLCBtdWxfemVybywgemVyb19hZGRdCiAgcmZsCgp0aGVvcmVtIHNlY29uZEJ1bXBXZWlnaHRlZE1vbWVudF90YWlsX2VxIChjIDogUGFyYW1ldGVycykgOgogICAgKOKIqyB4IGluIGMudXBwZXIgMC4uUmVhbC5leHAgYy5wdWxzZUxlbmd0aCwKICAgICAgeCBeIGMuZXhwb25lbnRzIDAgKgogICAgICAgIE5hdmllclN0b2tlcy5Mb2NhbGl6ZWRNb21lbnRSZXBhaXIuYnVtcCAoYy5sb3dlciAxKSAoYy51cHBlciAxKSB4KSA9CiAgICDiiKsgeCA6IOKEnSwgeCBeIGMuZXhwb25lbnRzIDAgKgogICAgICBOYXZpZXJTdG9rZXMuTG9jYWxpemVkTW9tZW50UmVwYWlyLmJ1bXAgKGMubG93ZXIgMSkgKGMudXBwZXIgMSkgeCA6PSBieQogIHJ3IFtpbnRlcnZhbEludGVncmFsLmludGVncmFsX2VxX2ludGVncmFsX29mX3N1cHBvcnRfc3Vic2V0XQogIGludHJvIHggaHgKICBoYXZlIGhiIDogTmF2aWVyU3Rva2VzLkxvY2FsaXplZE1vbWVudFJlcGFpci5idW1wIChjLmxvd2VyIDEpIChjLnVwcGVyIDEpIHgg4omgIDAgOj0gYnkKICAgIGludHJvIGh6ZXJvCiAgICBleGFjdCBoeCAoYnkgc2ltcCBbaHplcm9dKQogIGhhdmUgaHMgOj0gKE5hdmllclN0b2tlcy5Mb2NhbGl6ZWRNb21lbnRSZXBhaXIuYnVtcF90c3VwcG9ydF9zdWJzZXRfb3BlbgogICAgKGMubG93ZXIgMSkgKGMudXBwZXIgMSkgKGMubG93ZXJfbHRfdXBwZXIgMSkpIChzdWJzZXRfdHN1cHBvcnQgXyBoYikKICBleGFjdCDin6goYy5pbnRlcnZhbHNfc2VwYXJhdGVkIDAgMSAoYnkgZGVjaWRlKSkudHJhbnNfbHQgaHMuMSwKICAgIGhzLjIubGUudHJhbnMgKGMudXBwZXJfbHRfZW5kIDEpLmxl4p+pCgp0aGVvcmVtIGxvY2FsTWFzc191cHBlcl9lcV9zZWNvbmRfdGFpbCAoRiA6IFByb2ZpbGUpIChldGEgOiDihJ0pIDoKICAgIGxvY2FsTWFzcyBGIGV0YSAoRi5kYXRhLmNvcmUudXBwZXIgMCkgPQogICAgICAtKG1vbWVudFNjYWxlIEYuZGF0YS5jb3JlIDAgKiBzaGFwZSBldGEpICoKICAgICAgICAobG9jYWxTZWNvbmRSZXBhaXJDb2VmZmljaWVudCBGIGV0YSAqCiAgICAgICAgICAo4oirIHggOiDihJ0sIHggXiBGLmRhdGEuY29yZS5leHBvbmVudHMgMCAqCiAgICAgICAgICAgIE5hdmllclN0b2tlcy5Mb2NhbGl6ZWRNb21lbnRSZXBhaXIuYnVtcAogICAgICAgICAgICAgIChGLmRhdGEuY29yZS5sb3dlciAxKSAoRi5kYXRhLmNvcmUudXBwZXIgMSkgeCkpIDo9IGJ5CiAgbGV0IGMgOj0gRi5kYXRhLmNvcmUKICBjaGFuZ2UgbWFzc01vbWVudCBjIEYuYW1wIGV0YSAoYy5wdWxzZVN0YXJ0ICsgUmVhbC5sb2cgKGMudXBwZXIgMCkpID0gXwogIHJ3IFtsb2NhbE1hc3NfdGFpbF9pZGVudGl0eSBGIGV0YSAoYy51cHBlciAwKQogICAgKGZpcnN0X3VwcGVyX2d0X29uZSBjKS5sZSAoYy51cHBlcl9sdF9lbmQgMCkubGVdCiAgaGF2ZSBoaSA6ICjiiKsgeCBpbiBjLnVwcGVyIDAuLlJlYWwuZXhwIGMucHVsc2VMZW5ndGgsCiAgICAgIHggXiBjLmV4cG9uZW50cyAwICogcmFkaWFsUHVsc2UgYyBGLmFtcCBldGEgeCkgPQogICAgICBsb2NhbFNlY29uZFJlcGFpckNvZWZmaWNpZW50IEYgZXRhICoKICAgICAgICAo4oirIHggaW4gYy51cHBlciAwLi5SZWFsLmV4cCBjLnB1bHNlTGVuZ3RoLAogICAgICAgICAgeCBeIGMuZXhwb25lbnRzIDAgKgogICAgICAgICAgICBOYXZpZXJTdG9rZXMuTG9jYWxpemVkTW9tZW50UmVwYWlyLmJ1bXAgKGMubG93ZXIgMSkgKGMudXBwZXIgMSkgeCkgOj0gYnkKICAgIHJ3IFvihpAgaW50ZXJ2YWxJbnRlZ3JhbC5pbnRlZ3JhbF9jb25zdF9tdWxdCiAgICBhcHBseSBpbnRlcnZhbEludGVncmFsLmludGVncmFsX2NvbmdyCiAgICBpbnRybyB4IGh4CiAgICBoYXZlIGh4JyA6PSB1SWNjX29mX2xlIChjLnVwcGVyX2x0X2VuZCAwKS5sZSDilrggaHgKICAgIGRzaW1wIG9ubHkKICAgIHJ3IFt1cHBlclRhaWxfcmFkaWFsUHVsc2VfZXEgRiBldGEgeCBoeCcuMSBoeCcuMl0KICAgIHJpbmcKICBydyBbaGksIHNlY29uZEJ1bXBXZWlnaHRlZE1vbWVudF90YWlsX2VxIGNdCgp0aGVvcmVtIGxvY2FsTWFzc191cHBlcl9uZWdfb2Zfc2Vjb25kQ29lZmZfcG9zIChGIDogUHJvZmlsZSkgKGV0YSA6IOKEnSkKICAgIChoYzEgOiAwIDwgbG9jYWxTZWNvbmRSZXBhaXJDb2VmZmljaWVudCBGIGV0YSkgOgogICAgbG9jYWxNYXNzIEYgZXRhIChGLmRhdGEuY29yZS51cHBlciAwKSA8IDAgOj0gYnkKICBydyBbbG9jYWxNYXNzX3VwcGVyX2VxX3NlY29uZF90YWlsXQogIGV4YWN0IG11bF9uZWdfb2ZfbmVnX29mX3BvcwogICAgKG5lZ19uZWdfb2ZfcG9zIChtdWxfcG9zIChtb21lbnRTY2FsZV9wb3MgRi5kYXRhLmNvcmUgMCkgKHNoYXBlX3BvcyBldGEpKSkKICAgIChtdWxfcG9zIGhjMSAoc2Vjb25kQnVtcFdlaWdodGVkTW9tZW50X3BvcyBGLmRhdGEuY29yZSkpCgojcHJpbnQgYXhpb21zIE5TUlcuUDIucHJlZml4TV9wb3MKI3ByaW50IGF4aW9tcyBOU1JXLlAyLnJhZGlhbFB1bHNlX2JlZm9yZV9maXJzdF9sb3dlcgojcHJpbnQgYXhpb21zIE5TUlcuUDIubG9jYWxNYXNzX2xvd2VyX3BvcwojcHJpbnQgYXhpb21zIE5TUlcuUDIuc2Vjb25kQnVtcFdlaWdodGVkTW9tZW50X3BvcwojcHJpbnQgYXhpb21zIE5TUlcuUDIudXBwZXJUYWlsX3JhZGlhbFB1bHNlX2VxCiNwcmludCBheGlvbXMgTlNSVy5QMi5zZWNvbmRCdW1wV2VpZ2h0ZWRNb21lbnRfdGFpbF9lcQojcHJpbnQgYXhpb21zIE5TUlcuUDIubG9jYWxNYXNzX3VwcGVyX2VxX3NlY29uZF90YWlsCiNwcmludCBheGlvbXMgTlNSVy5QMi5sb2NhbE1hc3NfdXBwZXJfbmVnX29mX3NlY29uZENvZWZmX3BvcwoKZW5kIE5TUlcuUDIK","name":"P2_Z02_BoundaryMassSigns.lean"},{"sha256":"2af3c68a73530a97421b3bd971bae5e63bbfb60c1abb7f40361d6d347e875d4a","base64":"...","name":"P2_ZPLUS_J_FirstRepairAngularMoment.lean"},{"sha256":"fc887e9532abd28cbd0d8ea370a0c53f3111ea5f3031099d8050b6965f3ca33c","base64":"IiIiRGV2ZWxvcG1lbnQgb25seTogc291cmNlIC0+IGZyZXNoIFowMSAtPiBaMDIgLT4gY29uZGl0aW9uYWwgWitKOyBubyBhZG1pc3Npb24uIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGltcG9ydGxpYi51dGlsCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgcmUKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmltcG9ydCB6aXBmaWxlCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lLCB0aW1lem9uZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCkJBU0UgPSAiMTBkNWExZjUxN2I4NjY1MTM3ZjIzNTQ3NDZkNzA3NTJmOGVmNjdmZCIKSEVMUEVSX0JMT0IgPSAiNTE4ZGUwNjUzMGM0YzdjNTZlZGFmZGY1NzFlNWZiMmVmMGY4MDhiMyIKREVQRU5ERU5DSUVTID0gewogICAgIlAyX1owMV9Mb2NhbE1hc3NJZGVudGl0aWVzIjogImZmNzEwMTA2N2E5OGZiYWE4OTc0YzFhYTQwOWI4Mzc0MGNkMTdlYjRmMTc3OGYyMGNkYzIzYjQ0ZTJhNWMyNjciLAogICAgIlAyX1owMl9Cb3VuZGFyeU1hc3NTaWducyI6ICI3YjU3NjQ5MTE3OTAwNTRjYmJhOTgyYWNmZGU4ZWQ2NzM3MDVhMmQ2OGU4NWJjZDM0MTUwN2Q3ZTBiNWY4YzJjIiwKfQpNT0RVTEUgPSAiUDJfWlBMVVNfSl9GaXJzdFJlcGFpckFuZ3VsYXJNb21lbnQiClRIRU9SRU1TID0gKAogICAgImxvY2FsQW5ndWxhck1vbWVudF9wYXJ0aWFsX2lkZW50aXR5IiwgImxvY2FsQW5ndWxhck1vbWVudF90YWlsX2lkZW50aXR5IiwKICAgICJmaXJzdF91cHBlcl9sdF9zZWNvbmRfbG93ZXIiLCAic2Vjb25kRXhwb25lbnRfZXFfZmlyc3Rfc3ViX2xhbSIsCiAgICAiZnV0dXJlX3JhZGlhbFB1bHNlX2VxX3R3b19idW1wcyIsICJ3ZWlnaHRlZF9mdXR1cmVfaW50ZWdyYWxfZXFfdHdvX3RhaWxzIiwKICAgICJzZWNvbmRCdW1wX21hc3NUYWlsX3BvcyIsICJmaXJzdFJlcGFpcl9tYXNzX3plcm9fdGFpbF9iYWxhbmNlIiwKICAgICJmaXJzdEJ1bXBUYWlsX3Bvc19hbmRfZmlyc3RDb2VmZl9uZWciLCAiZmlyc3RCdW1wX2FuZ3VsYXJUYWlsX2xvd2VyIiwKICAgICJzZWNvbmRCdW1wX2FuZ3VsYXJUYWlsX3VwcGVyIiwgImFuZ3VsYXJNb21lbnRfcG9zX29mX2ZpcnN0UmVwYWlyX21hc3NfemVybyIsCikKU1RBTkRBUkRfQVhJT01TID0geyJwcm9wZXh0IiwgIkNsYXNzaWNhbC5jaG9pY2UiLCAiUXVvdC5zb3VuZCJ9CgoKZGVmIHNoYShwYXRoKToKICAgIHJldHVybiBoYXNobGliLnNoYTI1NihwYXRoLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCkKCgpkZWYgYmxvYihkYXRhKToKICAgIHJldHVybiBoYXNobGliLnNoYTEoYiJibG9iICIgKyBzdHIobGVuKGRhdGEpKS5lbmNvZGUoKSArIGIiXDAiICsgZGF0YSkuaGV4ZGlnZXN0KCkKCgpkZWYgcnVuX2NvbW1hbmQoY29tbWFuZCwgKiprd2FyZ3MpOgogICAgIiIiQm91bmRlZCB0ZXN0IGJvdW5kYXJ5OiBkbyBub3QgbW9ua2V5cGF0Y2ggc3RkbGliIHN1YnByb2Nlc3MgZ2xvYmFsbHkuIiIiCiAgICByZXR1cm4gc3VicHJvY2Vzcy5ydW4oY29tbWFuZCwgKiprd2FyZ3MpCgoKZGVmIHF1ZXJ5KHJvb3QsIGNvbW1hbmQsIGVudik6CiAgICByZXN1bHQgPSBydW5fY29tbWFuZChjb21tYW5kLCBjd2Q9cm9vdCwgZW52PWVudiwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgY2hlY2s9RmFsc2UpCiAgICBpZiByZXN1bHQucmV0dXJuY29kZToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIklERU5USVRZX1FVRVJZX0ZBSUxFRDogIiArIHJlcHIoY29tbWFuZCkpCiAgICByZXR1cm4gcmVzdWx0LnN0ZG91dAoKCmRlZiBpbnNwZWN0X2F4aW9tcyh0ZXh0KToKICAgIGlmICJzb3JyeUF4IiBpbiB0ZXh0OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNPUlJZX0FYSU9NX09CU0VSVkVEIikKICAgIHJlc3VsdCA9IHt9CiAgICBmb3Igc2hvcnQgaW4gVEhFT1JFTVM6CiAgICAgICAgbmFtZSA9ICJOU1JXLlAyLiIgKyBzaG9ydAogICAgICAgIG1hdGNoZXMgPSByZS5maW5kYWxsKHIiJyIgKyByZS5lc2NhcGUobmFtZSkgKyByIicgZGVwZW5kcyBvbiBheGlvbXM6IFxbKFteXF1dKilcXSIsIHRleHQpCiAgICAgICAgZW1wdHkgPSB0ZXh0LmNvdW50KCInIiArIG5hbWUgKyAiJyBkb2VzIG5vdCBkZXBlbmQgb24gYW55IGF4aW9tcyIpCiAgICAgICAgaWYgbGVuKG1hdGNoZXMpICsgZW1wdHkgIT0gMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQVhJT01fU1VSRkFDRV9NSVNTSU5HX09SX0RVUExJQ0FURUQ6ICIgKyBuYW1lKQogICAgICAgIHZhbHVlcyA9IFtwLnN0cmlwKCkgZm9yIHAgaW4gbWF0Y2hlc1swXS5zcGxpdCgiLCIpIGlmIHAuc3RyaXAoKV0gaWYgbWF0Y2hlcyBlbHNlIFtdCiAgICAgICAgaWYgbGVuKHZhbHVlcykgIT0gbGVuKHNldCh2YWx1ZXMpKSBvciBub3Qgc2V0KHZhbHVlcykgPD0gU1RBTkRBUkRfQVhJT01TOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBWElPTV9TVVJGQUNFX1VORVhQRUNURUQ6ICIgKyBuYW1lKQogICAgICAgICMgQXJpdGhtZXRpYyBoZWxwZXJzIG1heSBoYXZlIGEgc21hbGxlciBzdXJmYWNlOyB0aGUgc3RydWN0dXJhbCB0YXJnZXQKICAgICAgICAjIG11c3QgbWF0Y2ggdGhlIGFscmVhZHkgcmV2aWV3ZWQgc291cmNlLXByb29mIHN1cmZhY2UgZXhhY3RseS4KICAgICAgICBpZiBzaG9ydCA9PSBUSEVPUkVNU1stMV0gYW5kIHNldCh2YWx1ZXMpICE9IFNUQU5EQVJEX0FYSU9NUzoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiVEFSR0VUX0FYSU9NX1NVUkZBQ0VfVU5FWFBFQ1RFRCIpCiAgICAgICAgcmVzdWx0W25hbWVdID0gc29ydGVkKHZhbHVlcykKICAgIHJldHVybiByZXN1bHQKCgpkZWYgcnVuKHNvdXJjZSwgYmFzZSwgZGVwZW5kZW5jaWVzLCBwcm9vZiwgb3V0cHV0LCBleHBlY3RlZF9zaGEpOgogICAgc291cmNlLCBiYXNlLCBkZXBlbmRlbmNpZXMsIHByb29mLCBvdXRwdXQgPSAoCiAgICAgICAgcC5yZXNvbHZlKCkgZm9yIHAgaW4gKHNvdXJjZSwgYmFzZSwgZGVwZW5kZW5jaWVzLCBwcm9vZiwgb3V0cHV0KQogICAgKQogICAgYXJjaGl2ZSA9IFBhdGgoc3RyKG91dHB1dCkgKyAiLWV2aWRlbmNlLnppcCIpCiAgICBpZiAob3V0cHV0LmV4aXN0cygpIG9yIGFyY2hpdmUuZXhpc3RzKCkgb3IgcHJvb2YuaXNfcmVsYXRpdmVfdG8ob3V0cHV0KQogICAgICAgICAgICBvciBhbnkob3V0cHV0LmlzX3JlbGF0aXZlX3RvKHJvb3QpIGZvciByb290IGluIChzb3VyY2UsIGJhc2UsIGRlcGVuZGVuY2llcykpKToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPVVRQVVRfTVVTVF9CRV9ORVdfQU5EX09VVFNJREVfSU5QVVRfUk9PVFMiKQogICAgb3V0cHV0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9RmFsc2UpCiAgICBtZXRhID0gewogICAgICAgICJraW5kIjogIlpQTFVTX0pfREVWRUxPUE1FTlRfQ09NUElMRSIsICJjbGFpbV9hdXRob3JpdHkiOiAiTk9ORSIsCiAgICAgICAgImNsYWltX3N0YXR1cyI6ICJVTlZFUklGSUVEIiwgImNoZWNrX3N0YXR1cyI6ICJFUlJPUltOT1RfQ09NUExFVEVEXSIsCiAgICAgICAgInNjb3BlIjogIkZpcnN0LXdpbmRvdyBsb2NhbE1hc3M9MCBhbmQgYzE+MCBpbXBseSBsb2NhbEFuZ3VsYXJNb21lbnQ+MCBvbmx5IiwKICAgICAgICAibm9uX2NsYWltcyI6IFsiZXRhIHNpZ24iLCAiZXBzaWxvbiIsICJleGlzdGVuY2UgY29tcG9zaXRpb24iLCAiUHJvZmlsZS5KIGJyaWRnZSIsCiAgICAgICAgICAgICAgICAgICAgICAgInVuaXF1ZW5lc3MiLCAic2ltcGxlIHplcm8iLCAiUSBwb2xlIG9yIGxpbWl0IiwgImNsYWltIGFkbWlzc2lvbiJdLAogICAgICAgICJzdGFydGVkX3V0YyI6IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpLCAic3RhZ2VzIjoge30sCiAgICAgICAgImlucHV0cyI6IHt9LCAiZnJlc2hfYXJ0aWZhY3RzIjoge30sCiAgICB9CiAgICBlbnYgPSBvcy5lbnZpcm9uLmNvcHkoKQogICAgZW52WyJQQVRIIl0gPSAiL3Jvb3QvLmVsYW4vYmluOiIgKyBlbnYuZ2V0KCJQQVRIIiwgIiIpCiAgICBlbnYucG9wKCJMRUFOX1BBVEgiLCBOb25lKQogICAgZW52LnBvcCgiTEVBTl9TUkNfUEFUSCIsIE5vbmUpCgogICAgZGVmIHNhdmUoKToKICAgICAgICAob3V0cHV0IC8gInJ1bi1tZXRhZGF0YS5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKG1ldGEsIGluZGVudD0yKSwgZW5jb2Rpbmc9InV0Zi04IikKCiAgICBkZWYgc3RhZ2UobmFtZSwgY29tbWFuZCwgc3RhZ2VfZW52KToKICAgICAgICBpbmZvID0geyJjb21tYW5kIjogY29tbWFuZCwgImV4aXQiOiBOb25lfQogICAgICAgIG1ldGFbInN0YWdlcyJdW25hbWVdID0gaW5mbwogICAgICAgIG1ldGFbImFjdGl2ZV9zdGFnZSJdID0gbmFtZQogICAgICAgIHNhdmUoKQogICAgICAgIHN0YXJ0ZWQgPSB0aW1lLm1vbm90b25pYygpCiAgICAgICAgdHJ5OgogICAgICAgICAgICB3aXRoIChvdXRwdXQgLyAobmFtZSArICIuc3Rkb3V0LmxvZyIpKS5vcGVuKCJ4YiIpIGFzIG91dCwgXAogICAgICAgICAgICAgICAgICAgIChvdXRwdXQgLyAobmFtZSArICIuc3RkZXJyLmxvZyIpKS5vcGVuKCJ4YiIpIGFzIGVycjoKICAgICAgICAgICAgICAgIGluZm9bImV4aXQiXSA9IHJ1bl9jb21tYW5kKAogICAgICAgICAgICAgICAgICAgIGNvbW1hbmQsIGN3ZD1zb3VyY2UsIGVudj1zdGFnZV9lbnYsIHN0ZG91dD1vdXQsIHN0ZGVycj1lcnIsIGNoZWNrPUZhbHNlCiAgICAgICAgICAgICAgICApLnJldHVybmNvZGUKICAgICAgICBleGNlcHQgT1NFcnJvciBhcyBleGM6CiAgICAgICAgICAgIGluZm9bImludm9jYXRpb25fZXhjZXB0aW9uIl0gPSByZXByKGV4YykKICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKG5hbWUudXBwZXIoKSArICJfSU5WT0NBVElPTiIpIGZyb20gZXhjCiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgaW5mb1siZWxhcHNlZF9zZWNvbmRzIl0gPSB0aW1lLm1vbm90b25pYygpIC0gc3RhcnRlZAogICAgICAgICAgICBmb3Igc3RyZWFtIGluICgic3Rkb3V0IiwgInN0ZGVyciIpOgogICAgICAgICAgICAgICAgcGF0aCA9IG91dHB1dCAvIChuYW1lICsgIi4iICsgc3RyZWFtICsgIi5sb2ciKQogICAgICAgICAgICAgICAgaWYgcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgICAgICAgICBpbmZvW3N0cmVhbSArICJfc2hhMjU2Il0gPSBzaGEocGF0aCkKICAgICAgICAgICAgICAgICAgICBpbmZvW3N0cmVhbSArICJfYnl0ZXMiXSA9IHBhdGguc3RhdCgpLnN0X3NpemUKICAgICAgICAgICAgc2F2ZSgpCiAgICAgICAgcHJpbnQobmFtZSwgImV4aXQiLCBpbmZvWyJleGl0Il0sIGZsdXNoPVRydWUpCiAgICAgICAgaWYgaW5mb1siZXhpdCJdICE9IDA6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihuYW1lLnVwcGVyKCkgKyAiX0xFQU5fQ09NUElMRSIpCiAgICAgICAgaWYgaW5mb1sic3RkZXJyX2J5dGVzIl06CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihuYW1lLnVwcGVyKCkgKyAiX1NUREVSUl9SRVZJRVciKQogICAgICAgIGlmICJzb3JyeUF4IiBpbiAob3V0cHV0IC8gKG5hbWUgKyAiLnN0ZG91dC5sb2ciKSkucmVhZF9ieXRlcygpLmRlY29kZSgKICAgICAgICAgICAgICAgICJ1dGYtOCIsIGVycm9ycz0icmVwbGFjZSIpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IobmFtZS51cHBlcigpICsgIl9TT1JSWV9BWElPTV9PQlNFUlZFRCIpCgogICAgc2F2ZSgpCiAgICB0cnk6CiAgICAgICAgaWYgbm90IHJlLmZ1bGxtYXRjaChyIlswLTlhLWZdezY0fSIsIGV4cGVjdGVkX3NoYSkgb3Igc2hhKHByb29mKSAhPSBleHBlY3RlZF9zaGE6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlBST09GX1NIQV9NSVNNQVRDSCIpCiAgICAgICAgZm9yIG1vZHVsZSwgZGlnZXN0IGluIERFUEVOREVOQ0lFUy5pdGVtcygpOgogICAgICAgICAgICBpZiBzaGEoZGVwZW5kZW5jaWVzIC8gKG1vZHVsZSArICIubGVhbiIpKSAhPSBkaWdlc3Q6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJERVZFTE9QTUVOVF9ERVBFTkRFTkNZX1NIQV9NSVNNQVRDSDogIiArIG1vZHVsZSkKICAgICAgICBpZiBxdWVyeShiYXNlLCBbImdpdCIsICJyZXYtcGFyc2UiLCAiSEVBRCJdLCBlbnYpLmRlY29kZSgpLnN0cmlwKCkgIT0gQkFTRToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiREVQRU5ERU5DWV9CQVNFX01JU01BVENIIikKICAgICAgICBoZWxwZXJfZGF0YSA9IHF1ZXJ5KGJhc2UsIFsiZ2l0IiwgInNob3ciLCBCQVNFICsgIjpjb2xhYi9ydW4tYzAyLWRldmVsb3BtZW50LnB5Il0sIGVudikKICAgICAgICBpZiBibG9iKGhlbHBlcl9kYXRhKSAhPSBIRUxQRVJfQkxPQjoKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUElOTkVEX0lERU5USVRZX0hFTFBFUl9NSVNNQVRDSCIpCiAgICAgICAgaW5wdXRzLCBhcnRpZmFjdHMgPSBvdXRwdXQgLyAiaW5wdXRzIiwgb3V0cHV0IC8gImFydGlmYWN0cyIKICAgICAgICBpbnB1dHMubWtkaXIoKQogICAgICAgIGFydGlmYWN0cy5ta2RpcigpCiAgICAgICAgaGVscGVyX3BhdGggPSBpbnB1dHMgLyAicnVuLWMwMi1kZXZlbG9wbWVudC5weSIKICAgICAgICBoZWxwZXJfcGF0aC53cml0ZV9ieXRlcyhoZWxwZXJfZGF0YSkKICAgICAgICBkcml2ZXIgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkKICAgICAgICBvcmlnaW5hbHMgPSB7bW9kdWxlICsgIi5sZWFuIjogZGVwZW5kZW5jaWVzIC8gKG1vZHVsZSArICIubGVhbiIpCiAgICAgICAgICAgICAgICAgICAgIGZvciBtb2R1bGUgaW4gREVQRU5ERU5DSUVTfQogICAgICAgIG9yaWdpbmFscy51cGRhdGUoe01PRFVMRSArICIubGVhbiI6IHByb29mLCBkcml2ZXIubmFtZTogZHJpdmVyfSkKICAgICAgICBmb3IgbmFtZSwgb3JpZ2luYWwgaW4gb3JpZ2luYWxzLml0ZW1zKCk6CiAgICAgICAgICAgIHNodXRpbC5jb3B5ZmlsZShvcmlnaW5hbCwgaW5wdXRzIC8gbmFtZSkKICAgICAgICBtZXRhWyJpbnB1dHMiXSA9IHtwLm5hbWU6IHNoYShwKSBmb3IgcCBpbiBpbnB1dHMuaXRlcmRpcigpfQogICAgICAgIG1ldGFbImhlbHBlcl9naXRfYmxvYiJdID0gSEVMUEVSX0JMT0IKICAgICAgICBtZXRhWyJoZWxwZXJfYmFzZV9jb21taXQiXSA9IEJBU0UKICAgICAgICBtZXRhWyJwcm9vZl9zdGF0ZSJdID0gIlVOQ09NTUlUVEVEX0RFVkVMT1BNRU5UX0JZVEVTIgogICAgICAgIHNwZWMgPSBpbXBvcnRsaWIudXRpbC5zcGVjX2Zyb21fZmlsZV9sb2NhdGlvbigienBsdXNfc291cmNlX2NoZWNrcyIsIGhlbHBlcl9wYXRoKQogICAgICAgIGhlbHBlciA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoc3BlYykKICAgICAgICBwcmV2aW91c19ieXRlY29kZSA9IHN5cy5kb250X3dyaXRlX2J5dGVjb2RlCiAgICAgICAgdHJ5OgogICAgICAgICAgICBzeXMuZG9udF93cml0ZV9ieXRlY29kZSA9IFRydWUKICAgICAgICAgICAgc3BlYy5sb2FkZXIuZXhlY19tb2R1bGUoaGVscGVyKQogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIHN5cy5kb250X3dyaXRlX2J5dGVjb2RlID0gcHJldmlvdXNfYnl0ZWNvZGUKICAgICAgICBtZXRhWyJwcmVfc291cmNlX2lkZW50aXR5Il0gPSBoZWxwZXIuc291cmNlX2lkZW50aXR5KHNvdXJjZSwgZW52KQogICAgICAgIG1ldGFbIm5zcndfcHJlX29ic2VydmF0aW9uIl0gPSBoZWxwZXIubnNyd19vYnNlcnZhdGlvbihiYXNlLCBlbnYpCiAgICAgICAgc3RhZ2UoInNvdXJjZSIsIFsibGFrZSIsICJidWlsZCIsICIrTmF2aWVyU3Rva2VzLk91dGdvaW5nUHJvZmlsZSJdLCBlbnYpCiAgICAgICAgdGFyZ2V0ID0gc291cmNlIC8gaGVscGVyLlRBUkdFVF9BUlRJRkFDVAogICAgICAgIG1ldGFbInRhcmdldF9vbGVhbl9iZWZvcmVfc2hhMjU2Il0gPSBzaGEodGFyZ2V0KQogICAgICAgIHByb29mX2VudiA9IGVudi5jb3B5KCkKICAgICAgICBwcm9vZl9lbnZbIkxFQU5fUEFUSCJdID0gc3RyKGFydGlmYWN0cykKICAgICAgICBmb3IgbW9kdWxlIGluIFsqREVQRU5ERU5DSUVTLCBNT0RVTEVdOgogICAgICAgICAgICBmcmVzaCA9IGFydGlmYWN0cyAvIChtb2R1bGUgKyAiLm9sZWFuIikKICAgICAgICAgICAgaWYgZnJlc2guZXhpc3RzKCk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJBUlRJRkFDVF9OT1RfRlJFU0g6ICIgKyBtb2R1bGUpCiAgICAgICAgICAgIHN0YWdlKG1vZHVsZSwgWyJsYWtlIiwgImVudiIsICJsZWFuIiwgIi1SIiwgc3RyKGlucHV0cyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICItbyIsIHN0cihmcmVzaCksIHN0cihpbnB1dHMgLyAobW9kdWxlICsgIi5sZWFuIikpXSwgcHJvb2ZfZW52KQogICAgICAgICAgICBpZiBub3QgZnJlc2guaXNfZmlsZSgpIG9yIGZyZXNoLmlzX3N5bWxpbmsoKToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkZSRVNIX0FSVElGQUNUX01JU1NJTkc6ICIgKyBtb2R1bGUpCiAgICAgICAgICAgIG1ldGFbImZyZXNoX2FydGlmYWN0cyJdW2ZyZXNoLm5hbWVdID0gc2hhKGZyZXNoKQogICAgICAgIG1ldGFbImF4aW9tX3N1cmZhY2VzIl0gPSBpbnNwZWN0X2F4aW9tcygKICAgICAgICAgICAgKG91dHB1dCAvIChNT0RVTEUgKyAiLnN0ZG91dC5sb2ciKSkucmVhZF90ZXh0KGVuY29kaW5nPSJ1dGYtOCIpCiAgICAgICAgKQogICAgICAgIG1ldGFbInRhcmdldF9vbGVhbl9hZnRlcl9zaGEyNTYiXSA9IHNoYSh0YXJnZXQpCiAgICAgICAgbWV0YVsicG9zdF9zb3VyY2VfaWRlbnRpdHkiXSA9IGhlbHBlci5zb3VyY2VfaWRlbnRpdHkoc291cmNlLCBlbnYpCiAgICAgICAgbWV0YVsibnNyd19wb3N0X29ic2VydmF0aW9uIl0gPSBoZWxwZXIubnNyd19vYnNlcnZhdGlvbihiYXNlLCBlbnYpCiAgICAgICAgaWYgbWV0YVsicHJlX3NvdXJjZV9pZGVudGl0eSJdICE9IG1ldGFbInBvc3Rfc291cmNlX2lkZW50aXR5Il06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNPVVJDRV9JREVOVElUWV9EUklGVCIpCiAgICAgICAgaWYgbWV0YVsibnNyd19wcmVfb2JzZXJ2YXRpb24iXSAhPSBtZXRhWyJuc3J3X3Bvc3Rfb2JzZXJ2YXRpb24iXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTlNSV19PQlNFUlZBVElPTl9EUklGVCIpCiAgICAgICAgaWYgbWV0YVsidGFyZ2V0X29sZWFuX2JlZm9yZV9zaGEyNTYiXSAhPSBtZXRhWyJ0YXJnZXRfb2xlYW5fYWZ0ZXJfc2hhMjU2Il06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlNPVVJDRV9UQVJHRVRfRFJJRlQiKQogICAgICAgIGlmIG1ldGFbImlucHV0cyJdICE9IHtwLm5hbWU6IHNoYShwKSBmb3IgcCBpbiBpbnB1dHMuaXRlcmRpcigpfToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiQVJDSElWRURfSU5QVVRfRFJJRlQiKQogICAgICAgIGlmIG1ldGFbImZyZXNoX2FydGlmYWN0cyJdICE9IHtwLm5hbWU6IHNoYShwKSBmb3IgcCBpbiBhcnRpZmFjdHMuaXRlcmRpcigpfToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiRlJFU0hfQVJUSUZBQ1RfRFJJRlQiKQogICAgICAgIGlmIGFueShzaGEocCkgIT0gbWV0YVsiaW5wdXRzIl1bbmFtZV0gZm9yIG5hbWUsIHAgaW4gb3JpZ2luYWxzLml0ZW1zKCkpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFWEVDVVRJT05fSU5QVVRfRFJJRlQiKQogICAgICAgIGZvciBuYW1lLCBpbmZvIGluIG1ldGFbInN0YWdlcyJdLml0ZW1zKCk6CiAgICAgICAgICAgIGZvciBzdHJlYW0gaW4gKCJzdGRvdXQiLCAic3RkZXJyIik6CiAgICAgICAgICAgICAgICBpZiBzaGEob3V0cHV0IC8gKG5hbWUgKyAiLiIgKyBzdHJlYW0gKyAiLmxvZyIpKSAhPSBpbmZvW3N0cmVhbSArICJfc2hhMjU2Il06CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTE9HX0RSSUZUOiAiICsgbmFtZSkKICAgICAgICBtZXRhWyJjaGVja19zdGF0dXMiXSA9ICJQQVNTW0xPQ0FMX1BST09GX0NPTVBJTEU6UDJfWlBMVVNfSl0iCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBtZXRhWyJjaGVja19zdGF0dXMiXSA9ICJFUlJPUltaUExVU19KX0RFVkVMT1BNRU5UX0NPTVBJTEVdIgogICAgICAgIG1ldGFbImZhaWx1cmVfYm91bmRhcnkiXSA9IGdldGF0dHIoZXhjLCAiYm91bmRhcnkiLCBzdHIoZXhjKSkKICAgICAgICBtZXRhWyJlcnJvcl90eXBlIl0gPSB0eXBlKGV4YykuX19uYW1lX18KICAgICAgICBtZXRhWyJtYXRoZW1hdGljYWxfaW50ZXJwcmV0YXRpb24iXSA9ICJOT05FIgogICAgZmluYWxseToKICAgICAgICBtZXRhWyJmaW5pc2hlZF91dGMiXSA9IGRhdGV0aW1lLm5vdyh0aW1lem9uZS51dGMpLmlzb2Zvcm1hdCgpCiAgICAgICAgc2F2ZSgpCiAgICAgICAgd2l0aCB6aXBmaWxlLlppcEZpbGUoYXJjaGl2ZSwgIngiLCB6aXBmaWxlLlpJUF9ERUZMQVRFRCkgYXMgYnVuZGxlOgogICAgICAgICAgICBmb3IgcGF0aCBpbiBvdXRwdXQucmdsb2IoIioiKToKICAgICAgICAgICAgICAgIGlmIHBhdGguaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgICAgIGJ1bmRsZS53cml0ZShwYXRoLCBwYXRoLnJlbGF0aXZlX3RvKG91dHB1dCkuYXNfcG9zaXgoKSkKICAgICAgICBwcmludChtZXRhWyJjaGVja19zdGF0dXMiXSwgImNsYWltX2F1dGhvcml0eT1OT05FIiwgZmx1c2g9VHJ1ZSkKICAgICAgICBwcmludCgiQVJDSElWRSIsIGFyY2hpdmUsIHNoYShhcmNoaXZlKSwgYXJjaGl2ZS5zdGF0KCkuc3Rfc2l6ZSwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiBtZXRhCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBmb3IgbmFtZSBpbiAoInNvdXJjZSIsICJiYXNlIiwgImRlcGVuZGVuY2llcyIsICJwcm9vZiIsICJvdXRwdXQiKToKICAgICAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLSIgKyBuYW1lLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXByb29mLXNoYTI1NiIsIHJlcXVpcmVkPVRydWUpCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgb3V0Y29tZSA9IHJ1bihhcmdzLnNvdXJjZSwgYXJncy5iYXNlLCBhcmdzLmRlcGVuZGVuY2llcywgYXJncy5wcm9vZiwKICAgICAgICAgICAgICAgICAgYXJncy5vdXRwdXQsIGFyZ3MucHJvb2Zfc2hhMjU2KQogICAgcmFpc2UgU3lzdGVtRXhpdCgwIGlmIG91dGNvbWVbImNoZWNrX3N0YXR1cyJdLnN0YXJ0c3dpdGgoIlBBU1NbIikgZWxzZSAxKQo=","name":"run-zplus-j-development.py"}]''')
PIN = '10d5a1f517b8665137f2354746d70752f8ef67fd'
SOURCE = Path('/content/NavierStokesAndEuler')
assert SOURCE.is_dir(), 'PINNED_SOURCE_MISSING: restore Colab environment before compiling'
candidates = []
for pattern in ('nsrw-z04-clean-10d5a1f-*', 'nsrw-c04c-clean-10d5a1f-*'):
    for path in Path('/content').glob(pattern):
        result = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=path,
                                capture_output=True, check=False)
        if result.returncode == 0 and result.stdout.decode().strip() == PIN:
            candidates.append(path)
assert candidates, 'PINNED_BASE_MISSING: no install/clone performed'
BASE = sorted(candidates, key=lambda p: p.name)[-1]
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
INPUTS = Path('/content') / ('nsrw-zplus-j-inputs-' + stamp)
RUN = Path('/content') / ('nsrw-zplus-j-development-' + stamp)
INPUTS.mkdir(exist_ok=False)
for item in PAYLOAD:
    raw = base64.b64decode(item['base64'], validate=True)
    assert hashlib.sha256(raw).hexdigest() == item['sha256'], item['name']
    with (INPUTS / item['name']).open('xb') as out:
        out.write(raw)
proof = INPUTS / 'P2_ZPLUS_J_FirstRepairAngularMoment.lean'
digest = next(x['sha256'] for x in PAYLOAD if x['name'] == proof.name)
env = os.environ.copy()
env['PATH'] = '/root/.elan/bin:' + env.get('PATH', '')
command = [
    sys.executable, str(INPUTS / 'run-zplus-j-development.py'),
    '--source', str(SOURCE), '--base', str(BASE), '--dependencies', str(INPUTS),
    '--proof', str(proof), '--output', str(RUN), '--proof-sha256', digest,
]
print('DEVELOPMENT ONLY; claim_authority=NONE claim_status=UNVERIFIED', flush=True)
print('run=', RUN, 'base=', BASE, flush=True)
result = subprocess.run(command, env=env, check=False)
print('driver_exit=', result.returncode)
if (RUN / 'run-metadata.json').is_file():
    metadata = json.loads((RUN / 'run-metadata.json').read_text())
    print(json.dumps({k: metadata.get(k) for k in
        ('check_status', 'claim_authority', 'claim_status', 'failure_boundary',
         'axiom_surfaces', 'fresh_artifacts')}, indent=2))
    if result.returncode:
        for name in metadata.get('stages', {}):
            for stream in ('stdout', 'stderr'):
                path = RUN / (name + '.' + stream + '.log')
                if path.is_file() and path.stat().st_size:
                    print('\nRAW LOG PREVIEW', path.name)
                    print(path.read_bytes().decode('utf-8', errors='replace')[-8000:])
# Archive stays in this VM until explicitly saved and rehashed in D: evidence store.
assert result.returncode == 0, 'COMPILE NOT CLOSED: inspect preserved raw logs, no claim admission'
